# Disk convolution on a Colab A100 from VS Code

Experimental Triton forward and adjoint for the fixed PSF operator used by coronagraphoto. **GPU execution has not yet been validated.** Run the correctness cell before collecting timings. The CPU reference exercises JAX transformations; it does not execute Triton.

Keep this notebook in the local `output/disk-triton/` folder and open it in VS Code:

1. Use **Select Kernel > Colab** to connect to your A100 server. Use **New Colab Server** if you need to select a machine type. Start a fresh kernel before installing the pinned stack.
2. In VS Code's local Explorer, right-click the companion `disk-triton-colab.zip` and choose **Upload to Colab**. Select the same server as the notebook and wait for the upload notification.
3. Run the cells in order. The first cell only extracts the ZIP; it fails immediately if the upload is missing.

The notebook and saved cell outputs are local, but Python executes on the Colab server: `/content` is its filesystem. The extension's **Colab > Contents** view exposes these remote files. Use its refresh button after cells create files and **Download...** to copy files back to the local project. This notebook uses no browser-only upload/download widgets.

Results go in `/content/disk-triton/output/`; raw YIP downloads and the exported PSF cube go in `/content/disk-triton/data/`. These remote files survive a kernel restart on the same server, but must be downloaded before the server is deleted or recycled. The last cell creates a compact results ZIP and can optionally archive the data folder.

The bundle contains local wheels, tests, and scripts, so no unpublished branch needs to be fetched. See the [official extension guide](https://github.com/googlecolab/colab-vscode/wiki/User-Guide#working-with-files) for file transfer. Browser Colab also works: upload the ZIP through its Files sidebar instead.

In [ ]:
import zipfile
from pathlib import Path

archive = Path("/content/disk-triton-colab.zip")
root = Path("/content/disk-triton")
if not archive.is_file():
    raise FileNotFoundError(
        "In VS Code's local Explorer, right-click disk-triton-colab.zip > "
        "Upload to Colab, selecting this notebook's server. "
        "Wait for upload completion, then rerun. "
        "Expected: /content/disk-triton-colab.zip"
    )
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(root)
print(f"Extracted {archive.stat().st_size / 1024:.0f} KiB to {root}")
print(f"Found {len(list((root / 'wheels').glob('*.whl')))} package wheels.")

In [ ]:
%pip -q install uv
import subprocess
import sys

wheels = sorted(str(path) for path in (root / "wheels").glob("*.whl"))
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--system",
        "--python",
        sys.executable,
        "-r",
        str(root / "requirements-colab.txt"),
        "-c",
        str(root / "constraints-colab.txt"),
        *wheels,
    ],
    check=True,
)
# If JAX was imported before installation, restart the session before continuing.

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys
from pathlib import Path

root = Path("/content/disk-triton")
output = root / "output"
data = root / "data"
output.mkdir(parents=True, exist_ok=True)
data.mkdir(parents=True, exist_ok=True)
os.environ["JAX_ENABLE_X64"] = "true"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["YIPPY_CACHE_DIR"] = str(data / "yippy")

import jax  # noqa: E402 (env vars must be set before jax loads)

print(jax.devices())
assert jax.default_backend() == "gpu", (
    "Select a GPU runtime; do not benchmark CPU fallback."
)
assert "A100" in jax.devices()[0].device_kind, "This experiment targets an A100."
subprocess.run(["nvidia-smi"], check=True)
for package in ("jax", "jaxlib", "jax-triton", "triton", "equinox"):
    print(package, metadata.version(package))
(output / "environment.txt").write_text(
    subprocess.check_output(
        ["uv", "pip", "freeze", "--python", sys.executable], text=True
    )
)
print(f"Results: {output}")
print(f"Raw YIP cache: {os.environ['YIPPY_CACHE_DIR']}")

## Correctness gate

Tests include direct per-source sums, full and quarter cubes, odd/even grids, FP32/FP64 kernels, partial tile tails, candidate batching, the adjoint identity, nonlinear parameter gradients, Hessians, and coronagraphoto resampling/rotation. On an A100 all sixteen CUDA tests must run. Any failed test blocks performance conclusions. Numerical tolerances here are kernel checks, not a scientific noise budget.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", str(root / "tests"), "-q", "-rs"],
    check=True,
    cwd=root,
)

## Synthetic operator baseline

Compare the existing mapped contraction, explicit batched JAX, and Triton. Every result is synchronized. First-call cost is separate from warm timings. The cube is a dynamic, resident operand. Outputs include forward, adjoint, ring-rendering and value-plus-gradient times, FP64-reference errors, and hardware metadata.

The random PSFs are a bandwidth workload. These timings do not establish scientific accuracy or full disk-fit speed. Increase `--size` from 65 to 129 after the first run; avoid starting with a native 256-pixel cube.

In [ ]:
for dtype in ("float64", "float32"):
    subprocess.run(
        [
            sys.executable,
            str(root / "benchmarks/disk_convolution.py"),
            "--size",
            "65",
            "--batches",
            "1",
            "8",
            "32",
            "--dtype",
            dtype,
            "--repeats",
            "30",
            "--output",
            str(output / f"synthetic-{dtype}.json"),
        ],
        check=True,
    )

## Real PSFs and a complete Photomancy fit

This downloads EAC1 AAVC into `data/yippy/`, constructs a **64-pixel downsampled** PSF cube in an isolated temporary derived cache, renders a physical GraterDisk, and fits radius/inclination/position angle using coronagraphoto photon rates and a fixed Gaussian noise model. The raw download is reused on subsequent runs. The derived cache is temporary to avoid Yippy's performance-cache sampling collision; the cube itself is exported as `data/eac1-64.npy` for reuse and download.

Compare MAP parameters, covariance, evidence, and final gradient norm as well as time. The native-grid scientific convergence study remains separate.

The fit runs in FP64 and includes the MAP search and Hessian calculation. The default 100 steps are a budget, not a convergence guarantee: inspect the final gradient norm. For production disk fitting, also test other starting points and noise realizations.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(root / "benchmarks/photomancy_disk_fit.py"),
        "--psf-size",
        "64",
        "--image-size",
        "81",
        "--steps",
        "100",
        "--repeats",
        "3",
        "--backend",
        "triton",
        "--export-cube",
        str(data / "eac1-64.npy"),
        "--output",
        str(output / "photomancy-a100.json"),
    ],
    check=True,
)

In [ ]:
# Reuse the real cube to isolate the operator from disk rendering and optimization.
subprocess.run(
    [
        sys.executable,
        str(root / "benchmarks/disk_convolution.py"),
        "--cube",
        str(data / "eac1-64.npy"),
        "--batches",
        "1",
        "8",
        "32",
        "--dtype",
        "float64",
        "--repeats",
        "30",
        "--output",
        str(output / "real-cube-float64.json"),
    ],
    check=True,
)

In [ ]:
import json
import zipfile

# Set True to also archive raw YIP files and the exported cube for local storage.
# This can be large; download data/eac1-64.npy alone if that is all you need.
EXPORT_DATA = False

for path in sorted(output.glob("*.json")):
    print(path.name)
    result = json.loads(path.read_text())
    for row in result.get("rows", []):
        print(row)

results_zip = output / "disk-a100-results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as out:
    for path in [
        *sorted(output.glob("*.json")),
        root / "BUILD.json",
        output / "environment.txt",
    ]:
        out.write(path, path.name)
print(f"Results archive ready: {results_zip}")

if EXPORT_DATA:
    data_zip = output / "disk-a100-data.zip"
    print(f"Archiving {data}; large datasets may take several minutes.", flush=True)
    with zipfile.ZipFile(data_zip, "w", zipfile.ZIP_STORED) as out:
        for path in sorted(data.rglob("*")):
            if path.is_file():
                out.write(path, path.relative_to(root))
    print(f"Data archive ready: {data_zip} ({data_zip.stat().st_size / 1e9:.2f} GB)")

print("In VS Code: Colab > Contents > Refresh > disk-triton > output.")
print("Right-click each desired ZIP > Download... and choose the local project folder:")
print("output/disk-triton/")
print("Save this local notebook to preserve its cell outputs too.")

## Using the backend in an existing fit

Construct `YippyCoronagraph(..., disk_convolution_backend="triton")`, or replace an existing YippyCoronagraph's `disk_convolution_backend` leaf with `eqx.tree_at`. Use `"jax-batched"` for the portable flattened matrix path; `"jax"` retains the original default.

Pass the optical path as a field of an Equinox forward module, as in the fit script, and pass the logdensity as a dynamic argument to `eqx.filter_jit`. A Python closure containing the entire cube can embed it in the executable. The custom Triton operator supports derivatives of source brightness and higher source derivatives, but rejects differentiation or vmap over the PSF cube. Use JAX for variable optical parameters. Sharding and mixed GPU models are outside this prototype.